# 3dgs-lab-colab: 高速・永続保存版

動画/画像から3D Gaussian Splattingを作ります。前処理は **CUDA版PyCOLMAP**、学習は **gsplat/CUDA** を使用します。

高速化の中心は、(1) 240候補から120枚の有効キーフレームを選ぶ、(2) SfMの特徴抽出・照合をGPU化する、(3) Global Mapperを使う、(4) 8,000 step・60万Gaussian・SH degree 2へ抑える、の4点です。

成果物はGoogle Driveへ直接保存します。処理開始前にDrive書き込みを検証し、フレーム選別後・SfM後・学習4,000/8,000 stepの各時点で残します。

## 0. GPU確認（最初に必ず実行）
GPUランタイムでなければ、ここで止まります。ランタイム種別の選択ミスをCPU処理開始前に検出します。

In [ ]:
import os, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
assert gpu.returncode == 0 and gpu.stdout.strip(), (
    "GPUが見つかりません。ランタイム > ランタイムのタイプを変更 > GPU を選んでください。"
)
print("GPU:", gpu.stdout.strip())

## 1. 設定とGoogle Driveの保存確認
`balanced` が今回の推奨値です。`quick` は撮影確認、`quality` は品質優先です。実行ごとに時刻付きの新しい保存先を作り、過去の成果物を上書きしません。

In [ ]:
SCENE_NAME = "myscene"  #@param {type:"string"}
INPUT_TYPE = "video"  #@param ["video", "images"]
PROFILE = "balanced"  #@param ["quick", "balanced", "quality"]

PROFILES = {
    "quick":    dict(frames=80,  long_edge=1080, steps=5000,  splats=350000,  sh=1),
    "balanced": dict(frames=120, long_edge=1280, steps=8000,  splats=600000,  sh=2),
    "quality":  dict(frames=160, long_edge=1600, steps=12000, splats=1000000, sh=3),
}
cfg = PROFILES[PROFILE]
FRAMES_TARGET = cfg["frames"]
KEYFRAME_CANDIDATES = FRAMES_TARGET * 2
LONG_EDGE = cfg["long_edge"]
MAX_STEPS = cfg["steps"]
CAP_MAX_SPLATS = cfg["splats"]
SH_DEGREE = cfg["sh"]
SIFT_MAX_FEATURES = 4096
SEQUENTIAL_OVERLAP = 10

from datetime import datetime
RUN_ID = f"{SCENE_NAME}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
ROOT = f"/content/work/{RUN_ID}"
IMAGES_DIR = f"{ROOT}/images"
SPARSE_DIR = f"{ROOT}/sparse"
RESULT_DIR = f"/content/results/{RUN_ID}"
print(PROFILE, cfg, "run:", RUN_ID)

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")
DRIVE_RUN_DIR = Path(f"/content/drive/MyDrive/3dgs-lab/{SCENE_NAME}/{RUN_ID}")
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
probe = DRIVE_RUN_DIR / ".write_test.part"
probe.write_text("drive write ok\n", encoding="utf-8")
final_probe = DRIVE_RUN_DIR / "storage_verified.txt"
os.replace(probe, final_probe)
assert final_probe.read_text(encoding="utf-8") == "drive write ok\n"
print("永続保存先（書き込み検証済み）:", DRIVE_RUN_DIR)

## 2. セットアップ
PyTorch/gsplatに加え、Python 3.12対応のCUDA版PyCOLMAPを導入します。初回は数分かかります。

In [ ]:
import os, shutil, subprocess, sys

def run_cmd(cmd, check=True):
    print("$", " ".join(map(str, cmd)))
    return subprocess.run([str(x) for x in cmd], check=check)

run_cmd(["apt-get", "-qq", "update"])
run_cmd(["apt-get", "-qq", "install", "-y", "ffmpeg"])
run_cmd([sys.executable, "-m", "pip", "install", "--force-reinstall",
         "--index-url", "https://download.pytorch.org/whl/cu128", "torch"])
run_cmd([sys.executable, "-m", "pip", "install", "pycolmap-cuda12==4.1.1",
         "gsplat==1.5.3", "ninja", "opencv-python-headless", "imageio[ffmpeg]",
         "tqdm", "tyro>=0.8.8,!=1.0.9,!=1.0.10", "pyyaml", "matplotlib",
         "scikit-learn", "torchmetrics", "lpips", "piexif", "tensorboard",
         "viser", "splines"])

shutil.rmtree("/content/gsplat_repo", ignore_errors=True)
run_cmd(["git", "clone", "-q", "--branch", "v1.5.3", "--depth", "1",
         "https://github.com/nerfstudio-project/gsplat.git", "/content/gsplat_repo"])
run_cmd([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation",
         "git+https://github.com/rahul-goel/fused-ssim@328dc9836f513d00c4b5bc38fe30478b4435cbb5"])
run_cmd([sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/nerfstudio-project/nerfview@4538024fe0d15fd1a0e4d760f3695fc44ca72787"])

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch, gsplat, pycolmap
print("torch", torch.__version__, "CUDA build", torch.version.cuda, "available", torch.cuda.is_available())
print("gsplat", gsplat.__version__, "pycolmap", pycolmap.__version__)
assert torch.cuda.is_available(), "PyTorchからGPUが見えません"
has_cuda = getattr(pycolmap, "has_cuda", True)
has_cuda = has_cuda() if callable(has_cuda) else has_cuda
assert has_cuda is not False, "PyCOLMAPがCUDAビルドではありません"

## 3. 入力アップロード
動画なら1ファイル、画像なら複数画像またはzipを指定します。元動画はローカルにも残っている前提で、Driveには再開に必要な選別後画像から保存します。

In [ ]:
from google.colab import files
import glob, os, shutil, zipfile

uploaded = files.upload()
assert uploaded, "ファイルが選ばれていません"
uploaded_paths = []
for name, data in uploaded.items():
    dst = f"/content/upload_{os.path.basename(name)}"
    with open(dst, "wb") as stream:
        stream.write(data)
    uploaded_paths.append(dst)

VIDEO_PATH = None
RAW_IMAGES_DIR = f"{ROOT}/raw_images"
shutil.rmtree(RAW_IMAGES_DIR, ignore_errors=True)
os.makedirs(RAW_IMAGES_DIR, exist_ok=True)
if INPUT_TYPE == "video":
    assert len(uploaded_paths) == 1, "動画ファイルを1つだけアップロードしてください"
    VIDEO_PATH = uploaded_paths[0]
    print("video:", VIDEO_PATH, f"{os.path.getsize(VIDEO_PATH)/1e6:.1f} MB")
else:
    for path in uploaded_paths:
        if path.lower().endswith(".zip"):
            with zipfile.ZipFile(path) as archive:
                archive.extractall(RAW_IMAGES_DIR)
        else:
            shutil.copy2(path, RAW_IMAGES_DIR)
    print("uploaded image files:", len(glob.glob(f"{RAW_IMAGES_DIR}/**/*.*", recursive=True)))

## 4. 有効キーフレーム選別とDrive保存
時間範囲を均等に分けたうえで、各区間から **シャープネス70% + 適度な特徴点移動30%** のスコアが高いフレームを選びます。単純な等間隔200枚より、重複とブレを減らしながら撮影範囲を維持します。HDR動画はSDRへトーンマッピングします。

In [ ]:
import cv2, glob, json, math, numpy as np, os, shutil, subprocess, zipfile
from pathlib import Path
from PIL import Image, ImageOps

candidate_dir = f"{ROOT}/keyframe_candidates"
shutil.rmtree(candidate_dir, ignore_errors=True)
shutil.rmtree(IMAGES_DIR, ignore_errors=True)
os.makedirs(candidate_dir, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)

def scale_filter_expr(long_edge):
    return (f"scale=w='if(gt(iw,ih),min(iw,{long_edge}),-2)':"
            f"h='if(gt(iw,ih),-2,min(ih,{long_edge}))'")

if INPUT_TYPE == "video":
    probe = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0",
        "-show_entries", "stream=color_transfer", "-of", "json", VIDEO_PATH],
        capture_output=True, text=True, check=True)
    streams = json.loads(probe.stdout).get("streams", [{}])
    transfer = streams[0].get("color_transfer", "") if streams else ""
    dur = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1", VIDEO_PATH],
        capture_output=True, text=True, check=True)
    duration = float(dur.stdout.strip())
    fps = max(KEYFRAME_CANDIDATES / duration, 0.1)
    vf = f"fps={fps:.8f}"
    if transfer in {"arib-std-b67", "smpte2084"}:
        vf += (",zscale=t=linear:npl=100,format=gbrpf32le,zscale=p=bt709,"
               "tonemap=tonemap=hable:desat=0,zscale=t=bt709:m=bt709:r=tv,format=yuv420p")
        print("HDR -> SDR tone mapping:", transfer)
    vf += "," + scale_filter_expr(LONG_EDGE)
    subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "warning", "-y", "-i", VIDEO_PATH,
                    "-map_metadata", "-1", "-vf", vf, "-q:v", "2",
                    f"{candidate_dir}/frame_%05d.jpg"], check=True)
else:
    valid = {"jpg", "jpeg", "png", "bmp", "tif", "tiff"}
    sources = [p for p in sorted(glob.glob(f"{RAW_IMAGES_DIR}/**/*.*", recursive=True))
               if p.lower().rsplit(".", 1)[-1] in valid]
    for index, source in enumerate(sources, 1):
        with Image.open(source) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            image.thumbnail((LONG_EDGE, LONG_EDGE), Image.Resampling.LANCZOS)
            image.save(f"{candidate_dir}/frame_{index:05d}.jpg", quality=95)

paths = sorted(glob.glob(f"{candidate_dir}/frame_*.jpg"))
assert paths, "キーフレーム候補がありません"
orb = cv2.ORB_create(nfeatures=1000, fastThreshold=12)
features = {}
for path in paths:
    gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    scale = min(1.0, 360.0 / max(gray.shape))
    small = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    points, desc = orb.detectAndCompute(small, None)
    features[path] = dict(sharp=float(cv2.Laplacian(small, cv2.CV_64F).var()),
                          points=points or [], desc=desc, shape=small.shape)
sharp_values = np.array([features[p]["sharp"] for p in paths])
sharp_low, sharp_high = np.percentile(sharp_values, [10, 90])
matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

def motion_quality(previous, current):
    if previous is None or previous["desc"] is None or current["desc"] is None:
        return 0.0
    matches = matcher.match(previous["desc"], current["desc"])
    if len(matches) < 12:
        return 0.0
    matches = sorted(matches, key=lambda item: item.distance)[:120]
    p0 = np.float32([previous["points"][m.queryIdx].pt for m in matches])
    p1 = np.float32([current["points"][m.trainIdx].pt for m in matches])
    diagonal = math.hypot(*current["shape"])
    displacement = float(np.median(np.linalg.norm(p1 - p0, axis=1)) / max(diagonal, 1.0))
    moderate_motion = min(1.0, displacement / 0.02) * min(1.0, 0.12 / max(displacement, 1e-6))
    match_support = min(1.0, len(matches) / 60.0)
    return moderate_motion * match_support

keep = min(FRAMES_TARGET, len(paths))
selected = []
previous = None
for bucket in range(keep):
    start = round(bucket * len(paths) / keep)
    end = round((bucket + 1) * len(paths) / keep)
    choices = paths[start:max(start + 1, end)]
    def score(path):
        item = features[path]
        sharp = np.clip((item["sharp"] - sharp_low) / max(sharp_high - sharp_low, 1e-6), 0, 1)
        motion = 0.5 if previous is None else motion_quality(previous, item)
        return 0.70 * sharp + 0.30 * motion
    chosen = max(choices, key=score)
    selected.append(chosen)
    previous = features[chosen]
for index, source in enumerate(selected, 1):
    shutil.copy2(source, f"{IMAGES_DIR}/frame_{index:05d}.jpg")
shutil.rmtree(candidate_dir)
print(f"keyframes: {len(paths)} candidates -> {len(selected)} selected")

archive_part = DRIVE_RUN_DIR / "selected_images.zip.part"
archive_final = DRIVE_RUN_DIR / "selected_images.zip"
with zipfile.ZipFile(archive_part, "w", compression=zipfile.ZIP_STORED) as archive:
    for image_path in sorted(Path(IMAGES_DIR).glob("*.jpg")):
        archive.write(image_path, f"images/{image_path.name}")
os.replace(archive_part, archive_final)
with zipfile.ZipFile(archive_final) as archive:
    assert archive.testzip() is None and len(archive.namelist()) == len(selected)
print("Drive保存・検証完了:", archive_final, f"{archive_final.stat().st_size/1e6:.1f} MB")

## 5. CUDA PyCOLMAPでSfM
SIFT特徴抽出と照合をT4で実行し、Global Mapperで疎再構成します。Global Mapperの登録率が50%未満ならIncremental Mapperへ自動フォールバックし、登録枚数が多いモデルを採用します。完了後、カメラ姿勢と疎点群をDriveへ保存します。

In [ ]:
import json, math, os, pycolmap, shutil, zipfile
from pathlib import Path

DB_PATH = f"{ROOT}/colmap.db"
for path in [DB_PATH, SPARSE_DIR, f"{ROOT}/sparse_global", f"{ROOT}/sparse_incremental"]:
    if os.path.isdir(path):
        shutil.rmtree(path)
    elif os.path.exists(path):
        os.remove(path)
os.makedirs(SPARSE_DIR, exist_ok=True)

reader = pycolmap.ImageReaderOptions(camera_model="SIMPLE_RADIAL")
extract = pycolmap.FeatureExtractionOptions()
extract.max_image_size = LONG_EDGE
extract.use_gpu = True
extract.gpu_index = "0"
extract.sift.max_num_features = SIFT_MAX_FEATURES
pycolmap.extract_features(database_path=DB_PATH, image_path=IMAGES_DIR,
    camera_mode=pycolmap.CameraMode.SINGLE, reader_options=reader,
    extraction_options=extract, device=pycolmap.Device.cuda)

matching = pycolmap.FeatureMatchingOptions()
matching.use_gpu = True
matching.gpu_index = "0"
matching.max_num_matches = 16384
if INPUT_TYPE == "video":
    pairing = pycolmap.SequentialPairingOptions(overlap=SEQUENTIAL_OVERLAP, quadratic_overlap=True)
    pycolmap.match_sequential(database_path=DB_PATH, matching_options=matching,
        pairing_options=pairing, device=pycolmap.Device.cuda)
else:
    pycolmap.match_exhaustive(database_path=DB_PATH, matching_options=matching,
        device=pycolmap.Device.cuda)

if hasattr(pycolmap, "calibrate_view_graph"):
    try:
        pycolmap.calibrate_view_graph(DB_PATH)
    except Exception as error:
        print("view graph calibration warning:", error)

def registered_count(reconstruction):
    method = getattr(reconstruction, "num_reg_images", None)
    return int(method()) if callable(method) else len(reconstruction.images)

def best_reconstruction(models):
    values = list(models.values()) if hasattr(models, "values") else list(models)
    return max(values, key=registered_count) if values else None

try:
    global_models = pycolmap.global_mapping(database_path=DB_PATH, image_path=IMAGES_DIR,
        output_path=f"{ROOT}/sparse_global")
except Exception as error:
    print("Global Mapper failed; Incremental Mapperへフォールバック:", error)
    global_models = {}
best = best_reconstruction(global_models)
global_registered = registered_count(best) if best is not None else 0
total_images = len(list(Path(IMAGES_DIR).glob("*.jpg")))
if global_registered < max(3, math.ceil(total_images * 0.5)):
    print(f"Global Mapper {global_registered}/{total_images}: Incremental Mapperへフォールバック")
    incremental_models = pycolmap.incremental_mapping(database_path=DB_PATH, image_path=IMAGES_DIR,
        output_path=f"{ROOT}/sparse_incremental")
    incremental_best = best_reconstruction(incremental_models)
    if incremental_best is not None and registered_count(incremental_best) > global_registered:
        best = incremental_best
registered = registered_count(best) if best is not None else 0
assert best is not None and registered >= 3, "SfMに失敗しました。撮影経路・ブレ・動く被写体を確認してください。"
model_dir = Path(SPARSE_DIR) / "0"
model_dir.mkdir(parents=True, exist_ok=True)
best.write(model_dir)
ratio = registered / total_images
print(f"registered images: {registered}/{total_images} ({ratio:.1%})")
assert ratio >= 0.5, "登録率50%未満なので、低品質の学習を防ぐため停止しました"

sfm_part = DRIVE_RUN_DIR / "sfm_checkpoint.zip.part"
sfm_final = DRIVE_RUN_DIR / "sfm_checkpoint.zip"
with zipfile.ZipFile(sfm_part, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in model_dir.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(Path(ROOT)))
os.replace(sfm_part, sfm_final)
with zipfile.ZipFile(sfm_final) as archive:
    assert archive.testzip() is None and archive.namelist()
sfm_manifest = {"run_id": RUN_ID, "profile": PROFILE, "registered": registered,
                "total_images": total_images, "registration_ratio": ratio}
manifest_part = DRIVE_RUN_DIR / "sfm_manifest.json.part"
manifest_part.write_text(json.dumps(sfm_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
os.replace(manifest_part, DRIVE_RUN_DIR / "sfm_manifest.json")
print("Drive保存・検証完了:", sfm_final)

## 6. gsplat学習（CUDA/MCMC）
gsplat v1.5.3の学習本体を使いながら、データ読込部だけを現行PyCOLMAP API対応版へ差し替えます。Python 3.12で動かない旧SceneManager互換層は使いません。PLY保存先はDriveへ直結し、中間4,000 stepと最終8,000 step（balanced時）がランタイム消失後も残ります。評価レンダリングと頻繁なTensorBoard書込みは止め、学習本体へ時間を使います。

In [ ]:
import os, shutil, subprocess, sys, time
from pathlib import Path

run_cmd([sys.executable, "-m", "pip", "uninstall", "-y",
         "pycolmap", "pycolmap-cuda12"], check=False)
run_cmd([sys.executable, "-m", "pip", "install", "-q",
         "pycolmap-cuda12==4.1.1"])
run_cmd(["git", "-C", "/content/gsplat_repo", "fetch", "-q",
         "origin", "main", "--depth", "1"])
for rel in ["examples/datasets/colmap.py", "examples/exif.py"]:
    data = subprocess.check_output(
        ["git", "-C", "/content/gsplat_repo", "show", f"FETCH_HEAD:{rel}"]
    )
    Path("/content/gsplat_repo", rel).write_bytes(data)
# ColabのHugging Face datasetsと同名なので、ローカルpackageを明示する。
Path("/content/gsplat_repo/examples/datasets/__init__.py").touch()
run_cmd([sys.executable, "-c",
         f"import pycolmap; print('modern pycolmap', pycolmap.__version__); "
         f"pycolmap.Reconstruction(r'{ROOT}/sparse/0')"])

shutil.rmtree(RESULT_DIR, ignore_errors=True)
os.makedirs(RESULT_DIR, exist_ok=True)
drive_ply_dir = DRIVE_RUN_DIR / "ply"
drive_ply_dir.mkdir(parents=True, exist_ok=True)
os.symlink(drive_ply_dir, Path(RESULT_DIR) / "ply", target_is_directory=True)
mid_step = max(1000, MAX_STEPS // 2)
refine_stop = max(501, MAX_STEPS - 500)
cmd = [sys.executable, "simple_trainer.py", "mcmc",
       "--data_dir", ROOT, "--data_factor", "1", "--result_dir", RESULT_DIR,
       "--max_steps", str(MAX_STEPS), "--strategy.cap-max", str(CAP_MAX_SPLATS),
       "--strategy.refine-stop-iter", str(refine_stop), "--sh_degree", str(SH_DEGREE),
       "--eval_steps", "999999", "--save_steps", str(mid_step), str(MAX_STEPS),
       "--ply_steps", str(mid_step), str(MAX_STEPS), "--tb_every", "0",
       "--save_ply", "--disable_viewer", "--disable_video"]
print("$", " ".join(cmd))
started = time.time()
subprocess.run(cmd, cwd="/content/gsplat_repo/examples", check=True)
elapsed = time.time() - started
ply_files = sorted(drive_ply_dir.glob("point_cloud_*.ply"),
                   key=lambda path: int(path.stem.rsplit("_", 1)[-1]))
assert ply_files and ply_files[-1].stat().st_size > 1024, "Drive上に有効なPLYがありません"
print(f"training elapsed: {elapsed/60:.1f} min")
print("persistent PLY:", ply_files[-1], f"{ply_files[-1].stat().st_size/1e6:.1f} MB")

## 7. 成果物検証と任意ダウンロード
SHA-256とサイズをDriveのmanifestへ残します。`DOWNLOAD_NOW=True` にした場合だけブラウザ経由でもダウンロードします。ランタイムが切れても、表示されたDriveパスにPLYが残ります。

In [ ]:
DOWNLOAD_NOW = False  #@param {type:"boolean"}

import hashlib, json, os, shutil
from google.colab import files

final_ply = ply_files[-1]
digest = hashlib.sha256()
with open(final_ply, "rb") as stream:
    for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
        digest.update(chunk)
manifest = {
    "run_id": RUN_ID, "profile": PROFILE, "frames": FRAMES_TARGET,
    "long_edge": LONG_EDGE, "max_steps": MAX_STEPS,
    "cap_max_splats": CAP_MAX_SPLATS, "sh_degree": SH_DEGREE,
    "registered_images": registered, "total_images": total_images,
    "ply": final_ply.name, "bytes": final_ply.stat().st_size,
    "sha256": digest.hexdigest(), "training_seconds": elapsed
}
manifest_part = DRIVE_RUN_DIR / "result_manifest.json.part"
manifest_part.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
os.replace(manifest_part, DRIVE_RUN_DIR / "result_manifest.json")
assert final_ply.exists() and final_ply.stat().st_size == manifest["bytes"]
print("完了。ランタイム終了後も残る成果物:", final_ply)
print("SHA-256:", manifest["sha256"])
if DOWNLOAD_NOW:
    local_copy = f"/content/{SCENE_NAME}.ply"
    shutil.copy2(final_ply, local_copy)
    files.download(local_copy)

## 8. ローカルで見る
Google Driveの `MyDrive/3dgs-lab/<scene>/<run_id>/ply/` にある最新の `.ply` をMacへダウンロードし、`3dgs-lab/viewer/index.html` にドラッグ&ドロップしてください。

`selected_images.zip` と `sfm_checkpoint.zip` も残るため、学習だけやり直す場合に動画アップロードとSfMを繰り返す必要はありません。